# Module 1 — Patents (EPO)

Builds the empirical patent series from the raw EPO archive. For every patent
the novelty is the set of IPC codes it carries and the explorer is its first
inventor; the notebook accumulates both over the event sequence.

**Input** — one CSV per year, `YYYY.csv`, in `data/patents_EPO`, with columns
`publication.date` (`YYYYMMDD`), `inventor` (list-like string of dicts, each
with a `name`) and `ipc` (list-like string of IPC codes).

**Output** — into `outputs/`: `data_augmented_patents/` (per-year files with
the cumulative columns, plus the three frequency tables),
`delta_rows_patents.csv`, `reduced_patents_{log,lin}.csv`,
`epo_author_dates_1980_2020.pkl` and `epo_intervals.pkl`.

> This is the patents builder of the first release, unchanged. It is
> deliberately not the streaming rewrite used on the papers side: the code
> below is the one that produced the published series, and it reproduces
> `outputs/` byte for byte. Expect a long run — it parses the inventor field
> row by row.


In [1]:
# Repository bootstrap.
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "lib").is_dir() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT / "lib") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "lib"))

import pandas as pd

DATA_DIR = REPO_ROOT / "data"          # raw inputs, you provide these
OUTPUT_DIR = REPO_ROOT / "outputs"     # derived data, auto-generated

# Names used by the cells below, kept as in the first release.
data_dir_patents = DATA_DIR / "patents_EPO"
output_dir = OUTPUT_DIR
output_dir.mkdir(parents=True, exist_ok=True)

output_aug_dir = output_dir / "data_augmented_patents"
output_aug_dir.mkdir(parents=True, exist_ok=True)
data_aug_dir = output_aug_dir

start_year = 1980
end_year = 2020


## Exclude over-prolific first inventors

A first pass over the archive counts patents per (first inventor, year) and
collects the inventors above `max_patents_per_year`, which are institutional
filers rather than individuals. They are dropped from every year.


In [2]:
import pandas as pd
from pathlib import Path
from tqdm import tqdm

# === Paths ===

start_year = 1980
end_year = 2020
max_patents_per_year = 50

excluded_authors = set()
all_first_authors = set()
first_author_total_counts = {}
total_patents_before = 0

def safe_eval_list(s):
    try:
        val = eval(s)
        return val if isinstance(val, list) else []
    except Exception:
        return []


def extract_first_author(raw_inventor_cell):
    inventor_list = safe_eval_list(raw_inventor_cell)
    inventor_list = [eval(x) if isinstance(x, str) else x for x in inventor_list]
    for d in inventor_list:
        if isinstance(d, dict):
            name = d.get("name", "").strip()
            if name:
                return name
    return ""


for path in sorted(data_dir_patents.glob("*.csv")):
    if not path.stem.isdigit():
        continue

    year = int(path.stem)
    if year < start_year or year > end_year:
        continue

    df = pd.read_csv(path, usecols=["inventor"])
    total_patents_before += len(df)
    counts = {}

    for raw_inventor in tqdm(df["inventor"], total=len(df), desc=f"Scanning {path.name}", ncols=80):
        first_author = extract_first_author(raw_inventor)
        if not first_author:
            continue
        all_first_authors.add(first_author)
        first_author_total_counts[first_author] = first_author_total_counts.get(first_author, 0) + 1
        counts[first_author] = counts.get(first_author, 0) + 1

    for name, c in counts.items():
        if c > max_patents_per_year:
            excluded_authors.add(name)

removed = len(excluded_authors)
found = len(all_first_authors)
remaining = found - removed

print(f"Total first authors found: {found:,}")
print(f"Total first authors removed: {removed:,}")
print(f"Total first authors after removal: {remaining:,}")

# Count patents before/after exclusion by first author
total_patents_removed = sum(first_author_total_counts.get(a, 0) for a in excluded_authors)
total_patents_after = total_patents_before - total_patents_removed
print(f"Total patents before exclusion: {total_patents_before:,}")
print(f"Total patents removed: {total_patents_removed:,}")
print(f"Total patents after exclusion: {total_patents_after:,}")


Scanning 1980.csv:   0%|                               | 0/6098 [00:00<?, ?it/s]

Scanning 1980.csv:  45%|███████▋         | 2737/6098 [00:00<00:00, 27366.50it/s]

Scanning 1980.csv:  90%|███████████████▎ | 5474/6098 [00:00<00:00, 26938.28it/s]

Scanning 1980.csv: 100%|█████████████████| 6098/6098 [00:00<00:00, 27187.25it/s]

Scanning 1981.csv:   0%|                               | 0/9995 [00:00<?, ?it/s]

...

Scanning 2020.csv:  94%|██████████████ | 66861/71007 [00:04<00:00, 15965.08it/s]

Scanning 2020.csv:  96%|██████████████▍| 68459/71007 [00:04<00:00, 15314.33it/s]

Scanning 2020.csv:  99%|██████████████▊| 70037/71007 [00:04<00:00, 15447.76it/s]

Scanning 2020.csv: 100%|███████████████| 71007/71007 [00:04<00:00, 15715.98it/s]

Total first authors found: 894,634
Total first authors removed: 7
Total first authors after removal: 894,627
Total patents before exclusion: 1,443,870
Total patents removed: 10,379
Total patents after exclusion: 1,433,491


## Main pass

Walks the events in date order, accumulating unique IPC combinations and unique
inventors, and writes one augmented CSV per year plus the frequency tables.


In [3]:
import pandas as pd
from pathlib import Path
from tqdm import tqdm
from collections import defaultdict

# === Paths ===
# Read merged yearly files from data/ and write augmented output to data_augmented_patents/

start_year = 1980
end_year = 2020

# === Global counters for cumulative columns ===
# These counters increase across years to build cumulative series
unique_ipc_sets = set()
unique_authors = set()
unique_first_authors = set()
patent_counter = 1
ipc_counter = 0
author_counter = 0
first_author_counter = 0

# === Frequency tables ===
# Frequencies over the full time span (not per-year)
ipc_freq = defaultdict(int)
author_freq = defaultdict(int)
first_author_freq = defaultdict(int)

def normalize_ipc_list(ipc_list):
    return tuple(sorted(ipc_list))

def safe_eval_list(s):
    try:
        val = eval(s)
        return val if isinstance(val, list) else []
    except Exception:
        return []

for path in sorted(data_dir_patents.glob("*.csv")):
    if not path.stem.isdigit():
        continue

    year = int(path.stem)
    if year < start_year or year > end_year:
        continue

    print(f"Processing file: {path.name}")
    df = pd.read_csv(path)

    # Sort by publication date for cumulative counters
    df['publication.date'] = pd.to_datetime(df['publication.date'], format="%Y%m%d", errors='coerce')
    df = df.sort_values(by='publication.date').reset_index(drop=True)

    cleaned_rows = []
    ipc_uniques = []
    author_uniques = []
    first_author_uniques = []

    # Iterate row by row to build cumulative counters in chronological order
    for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Processing {path.name}", ncols=80):
        inventor_list = safe_eval_list(row.get('inventor', '[]'))
        inventor_list = [eval(x) if isinstance(x, str) else x for x in inventor_list]

        # Skip rows with missing/invalid inventors (empty or placeholder names)
        if not inventor_list or any(
            isinstance(d, dict) and (
                d.get("name", "").lower().startswith("the designation of the inventor") or
                d.get("name", "").strip() == ""
            )
            for d in inventor_list
        ):
            continue

        # Extract inventor names and keep the first author separately
        inventor_names = [d.get("name", "").strip() for d in inventor_list if d.get("name")]

        if not inventor_names:
            continue
        first_author = inventor_names[0]
        if excluded_authors and first_author in excluded_authors:
            continue



        # Update unique author counters (cumulative across all years)
        new_authors = [name for name in inventor_names if name not in unique_authors]
        if new_authors:
            unique_authors.update(new_authors)
            author_counter += len(new_authors)
        author_uniques.append(author_counter)

        # Update unique first author counters (cumulative across all years)
        if first_author not in unique_first_authors:
            unique_first_authors.add(first_author)
            first_author_counter += 1
        first_author_uniques.append(first_author_counter)

        # Update unique IPC combinations (cumulative across all years)
        # IPC combinations are treated as unordered sets (sorted to normalize)
        ipc_list = safe_eval_list(row.get('ipc', '[]'))
        ipc_key = normalize_ipc_list(ipc_list)
        if ipc_key not in unique_ipc_sets:
            unique_ipc_sets.add(ipc_key)
            ipc_counter += 1
        ipc_uniques.append(ipc_counter)

        # Update frequency tables (global counts over full period)
        ipc_freq[ipc_key] += 1
        for name in inventor_names:
            author_freq[name] += 1
        if inventor_names:
            first_author_freq[inventor_names[0]] += 1

        cleaned_rows.append(row)

    cleaned_df = pd.DataFrame(cleaned_rows).reset_index(drop=True)
    num_rows = len(cleaned_df)
    cleaned_df['number of patents'] = list(range(patent_counter, patent_counter + num_rows))
    cleaned_df['unique ipc combinations'] = ipc_uniques
    cleaned_df['unique authors'] = author_uniques
    cleaned_df['unique authors (only first)'] = first_author_uniques
    patent_counter += num_rows

    out_path = output_aug_dir / path.name
    cleaned_df.to_csv(out_path, index=False)

    print(
        f"Finished {path.name} — Rows kept: {num_rows}, "
        f"Total IPCs: {ipc_counter}, Total Authors: {author_counter}, Total First Authors: {first_author_counter}"
    )

# === Save frequency tables ===
print("Writing frequency tables...")

df_ipc = pd.DataFrame({'element': list(map(str, ipc_freq.keys())), 'frequency': list(ipc_freq.values())})
df_authors = pd.DataFrame({'element': list(author_freq.keys()), 'frequency': list(author_freq.values())})
df_first_authors = pd.DataFrame({'element': list(first_author_freq.keys()), 'frequency': list(first_author_freq.values())})

df_ipc.to_csv(output_aug_dir / "ipc_combos_freq.csv", index=False)
df_authors.to_csv(output_aug_dir / "unique_authors_freq.csv", index=False)
df_first_authors.to_csv(output_aug_dir / "unique_first_authors_freq.csv", index=False)

print(f"Done. Augmented files and frequencies saved in: {output_aug_dir.resolve()}")


Processing file: 1980.csv


Processing 1980.csv:   0%|                             | 0/6098 [00:00<?, ?it/s]

Processing 1980.csv:  11%|█▉               | 694/6098 [00:00<00:00, 6939.59it/s]

Processing 1980.csv:  24%|███▊            | 1443/6098 [00:00<00:00, 7260.51it/s]

Processing 1980.csv:  36%|█████▋          | 2170/6098 [00:00<00:00, 5949.13it/s]

...

Processing 2020.csv:  99%|█████████████▊| 70170/71007 [00:15<00:00, 5828.68it/s]

Processing 2020.csv: 100%|█████████████▉| 70759/71007 [00:15<00:00, 5823.02it/s]

Processing 2020.csv: 100%|██████████████| 71007/71007 [00:15<00:00, 4671.88it/s]

Finished 2020.csv — Rows kept: 70046, Total IPCs: 706996, Total Authors: 2034213, Total First Authors: 893828
Writing frequency tables...


Done. Augmented files and frequencies saved in: ./outputs/data_augmented_patents


## Unique IPC codes


In [4]:
import ast

# Count unique IPC codes (not IPC combinations) across all yearly augmented files
unique_ipc_codes = set()

for path in sorted(data_aug_dir.glob("*.csv")):
    if not path.stem.isdigit():
        continue

    df = pd.read_csv(path, usecols=['ipc'])
    for value in df['ipc'].dropna():
        if isinstance(value, str):
            try:
                ipc_list = ast.literal_eval(value)
            except (ValueError, SyntaxError):
                continue
        elif isinstance(value, (list, tuple, set)):
            ipc_list = value
        else:
            continue

        for ipc_code in ipc_list:
            if pd.notna(ipc_code):
                unique_ipc_codes.add(str(ipc_code))

print(f"Total number of unique IPC codes in the dataset: {len(unique_ipc_codes)}")

Total number of unique IPC codes in the dataset: 71782


## Yearly deltas: events and active first inventors per year


In [5]:
import pandas as pd
from pathlib import Path
import numpy as np

# === Directory with augmented yearly CSVs ===

# Lists for cumulative values by year
years = []
num_rows_cumulative = []
num_ipc_combos = []
num_authors = []
num_first_authors = []

for path in sorted(data_aug_dir.glob("*.csv")):
    if not path.stem.isdigit():
        continue

    year = int(path.stem)
    df = pd.read_csv(path)
    last_row = df.iloc[-1]

    rows = int(last_row['number of patents'])

    years.append(year)
    num_rows_cumulative.append(rows)
    num_ipc_combos.append(int(last_row['unique ipc combinations']))
    num_authors.append(int(last_row['unique authors']))
    num_first_authors.append(int(last_row['unique authors (only first)']))

delta_rows = np.diff(num_rows_cumulative, prepend=0)
delta_rows_df = pd.DataFrame({'delta_rows': delta_rows, 'num_first_authors': num_first_authors})
delta_rows_df.to_csv(output_dir / 'delta_rows_patents.csv', index=False)


## Full series and reduced samples

The cumulative series is subsampled on a log and a linear grid and written as
`reduced_patents_log.csv` and `reduced_patents_lin.csv`, the files every
downstream module reads.


In [6]:
import pandas as pd
from pathlib import Path

# === Directory with augmented yearly CSVs ===

# === Full series (row by row) ===
years = []
all_rows = []
all_ipc = []
all_authors = []
all_first_authors = []

for path in sorted(data_aug_dir.glob("*.csv")):
    if not path.stem.isdigit():
        continue

    year = int(path.stem)
    df = pd.read_csv(path)

    years.extend([year] * len(df))
    all_rows.extend(df['number of patents'].tolist())
    all_ipc.extend(df['unique ipc combinations'].tolist())
    all_authors.extend(df['unique authors'].tolist())
    all_first_authors.extend(df['unique authors (only first)'].tolist())


In [7]:
import pandas as pd
from pathlib import Path

# === Frequency tables ===
freq_dir = output_aug_dir

# Read the three frequency files
ipc_freq_path = freq_dir / "ipc_combos_freq.csv"
authors_freq_path = freq_dir / "unique_authors_freq.csv"
first_authors_freq_path = freq_dir / "unique_first_authors_freq.csv"

df_ipc = pd.read_csv(ipc_freq_path)
df_authors = pd.read_csv(authors_freq_path)
df_first_authors = pd.read_csv(first_authors_freq_path)

# Sort by descending frequency (rank)
ipc_freq_sorted = df_ipc['frequency'].sort_values(ascending=False).reset_index(drop=True)
authors_freq_sorted = df_authors['frequency'].sort_values(ascending=False).reset_index(drop=True)
first_authors_freq_sorted = df_first_authors['frequency'].sort_values(ascending=False).reset_index(drop=True)


In [8]:
import numpy as np
from pathlib import Path

# Number of desired points
n_points = 10**4
N = len(years)

# Indici logaritmici (da 1 a N, poi -1 perché Python è 0-based)
log_indices = np.unique(np.logspace(0, np.log10(N), n_points, dtype=int) - 1)

# Indici lineari
lin_indices = np.linspace(0, N-1, n_points, dtype=int)

# Ora puoi sottocampionare i vettori
years_log = np.array(years)[log_indices]
all_ipc_log = np.array(all_ipc)[log_indices]
all_first_authors_log = np.array(all_first_authors)[log_indices]
all_rows_log = np.array(all_rows)[log_indices]

years_lin = np.array(years)[lin_indices]
all_ipc_lin = np.array(all_ipc)[lin_indices]
all_first_authors_lin = np.array(all_first_authors)[lin_indices]
all_rows_lin = np.array(all_rows)[lin_indices]

import pandas as pd
df_log = pd.DataFrame({
    'years': years_log,
    'all_ipc': all_ipc_log,
    'all_first_authors': all_first_authors_log,
    'all_rows': all_rows_log
})
df_log.to_csv(output_dir / "reduced_patents_log.csv", index=False)
df_lin = pd.DataFrame({
    'years': years_lin,
    'all_ipc': all_ipc_lin,
    'all_first_authors': all_first_authors_lin,
    'all_rows': all_rows_lin
})
df_lin.to_csv(output_dir / "reduced_patents_lin.csv", index=False)

## Per-inventor event timelines

`epo_author_dates_1980_2020.pkl`: for each first inventor, the list of their
events with date and novelty type. Used by Module 2 for the productivity
figure and the activity heatmap.


In [9]:
import pandas as pd
from pathlib import Path
from collections import defaultdict
from tqdm import tqdm
import ast
import json
import re
import pickle


first_year = 1980
last_year = 2020

# --- Helpers ---

def parse_inventors_field(inv_str):
    '''Return list of inventor names from a messy stringified list.'''
    if not isinstance(inv_str, str):
        return []

    # Try JSON
    try:
        parsed = json.loads(inv_str)
        if isinstance(parsed, list):
            names = [inv.get("name", "").strip() for inv in parsed if isinstance(inv, dict)]
            names = [n for n in names if n]
            if names:
                return names
    except Exception:
        pass

    # Try literal_eval
    try:
        parsed = ast.literal_eval(inv_str)
        if isinstance(parsed, list):
            names = [inv.get("name", "").strip() for inv in parsed if isinstance(inv, dict)]
            names = [n for n in names if n]
            if names:
                return names
    except Exception:
        pass

    # Fallback: regex for 'name': '...'
    names = re.findall(r"'name'\s*:\s*'([^']+)'", inv_str)
    return [n.strip() for n in names if n.strip()]

# --- Step 1: collect all patents for global novelty ---
all_patents = []  # entries: {date, author, ipc}

for path in sorted(data_aug_dir.glob("*.csv")):
    if not path.stem.isdigit():
        continue

    year = int(path.stem)
    if not (first_year <= year <= last_year):
        continue

    df = pd.read_csv(path)
    if not {"publication.date", "inventor", "ipc"}.issubset(df.columns):
        continue

    df["publication.date"] = pd.to_datetime(df["publication.date"], errors="coerce")

    for _, row in df.iterrows():
        if pd.isna(row["publication.date"]):
            continue

        inventors_list = parse_inventors_field(row["inventor"])
        if not inventors_list:
            continue

        first_author = inventors_list[0].strip()
        if not first_author:
            continue

        # Parse IPC list
        try:
            ipc_list = ast.literal_eval(row["ipc"]) if isinstance(row["ipc"], str) else row["ipc"]
            if not isinstance(ipc_list, list):
                continue
        except Exception:
            continue

        all_patents.append({
            "date": row["publication.date"],
            "author": first_author,
            "ipc": tuple(sorted(ipc_list))
        })

print(f"Total patents collected: {len(all_patents):,}")

# --- Step 2: sort by date (required for global novelty) ---
all_patents = sorted(all_patents, key=lambda x: x["date"])

# --- Step 3: assign novelty types (global vs individual) ---

global_seen = set()
individual_seen = defaultdict(set)

author_dates = defaultdict(list)

for entry in all_patents:
    author = entry["author"]
    date = entry["date"]
    ipc_combo = entry["ipc"]

    if ipc_combo not in global_seen:
        novelty = "global_novelty"
        global_seen.add(ipc_combo)
        individual_seen[author].add(ipc_combo)
    else:
        if ipc_combo not in individual_seen[author]:
            novelty = "individual_novelty"
            individual_seen[author].add(ipc_combo)
        else:
            novelty = "no_novelty"

    author_dates[author].append({
        "date": date,
        "ipc": list(ipc_combo),
        "novelty_type": novelty
    })

# --- Step 4: enrich with time features and cumulative novelty counters ---
max_years = last_year - first_year

for author, events in author_dates.items():
    if not events:
        continue

    events = sorted(events, key=lambda x: x["date"])
    t0 = pd.to_datetime(events[0]["date"])

    cumulative_individual = 0
    cumulative_global = 0

    for i, ev in enumerate(events):
        t = pd.to_datetime(ev["date"])

        if ev["novelty_type"] == "global_novelty":
            cumulative_global += 1
            cumulative_individual += 1
        elif ev["novelty_type"] == "individual_novelty":
            cumulative_individual += 1

        ev["cumulative_individual_novelty"] = cumulative_individual
        ev["cumulative_global_novelty"] = cumulative_global

        ev["year"] = t.year
        ev["year_relative"] = t.year - t0.year

        rel_years = (t - t0).total_seconds() / (365.25 * 24 * 3600)
        ev["relative_year"] = rel_years
        ev["cumulative_count"] = i + 1

        day_of_year = t.timetuple().tm_yday
        abs_year_decimal = t.year + (day_of_year - 1) / 365.25
        ev["abs_year_decimal"] = abs_year_decimal

        scaled = (abs_year_decimal - first_year) * (max_years / (last_year - first_year))
        ev["scaled_year_global"] = scaled

    author_dates[author] = events

# --- Save ---
output_path = output_dir / f"epo_author_dates_{first_year}_{last_year}.pkl"
with open(output_path, "wb") as f:
    pickle.dump(author_dates, f, protocol=pickle.HIGHEST_PROTOCOL)

print(f"Saved author dates to {output_path}")


Total patents collected: 1,420,340


Saved author dates to ./outputs/epo_author_dates_1980_2020.pkl


## Inter-event times

`epo_intervals.pkl`: the gaps between consecutive events of the same inventor,
split by novelty type. Feeds the inter-event statistics quoted in the SI.


In [10]:
import numpy as np
import pandas as pd
import pickle
from pathlib import Path

input_path = output_dir / "epo_author_dates_1980_2020.pkl"

with open(input_path, "rb") as f:
    author_dates = pickle.load(f)


def compute_intervals_with_dates(events_list, author_name):
    '''Compute inter-event times with start/end dates and author name.'''
    if len(events_list) < 2:
        return []

    dates = sorted([pd.to_datetime(ev["date"]) for ev in events_list])

    intervals = []
    for d1, d2 in zip(dates[:-1], dates[1:]):
        dt = (d2 - d1).days / 365.25
        if 0 < dt < 50:
            intervals.append({
                "interval_years": dt,
                "start_date": d1,
                "end_date": d2,
                "author": author_name
            })

    return intervals

all_intervals = []
global_only_intervals = []
novelties_intervals = []
no_novelty_intervals = []

for author, events in author_dates.items():
    all_intervals.extend(compute_intervals_with_dates(events, author))

    events_global = [ev for ev in events if ev["novelty_type"] == "global_novelty"]
    global_only_intervals.extend(compute_intervals_with_dates(events_global, author))

    events_novel = [ev for ev in events if ev["novelty_type"] in ("global_novelty", "individual_novelty")]
    novelties_intervals.extend(compute_intervals_with_dates(events_novel, author))

    events_none = [ev for ev in events if ev["novelty_type"] == "no_novelty"]
    no_novelty_intervals.extend(compute_intervals_with_dates(events_none, author))

intervals_dict = {
    "all_intervals": all_intervals,
    "global_only_intervals": global_only_intervals,
    "novelties_intervals": novelties_intervals,
    "no_novelty_intervals": no_novelty_intervals,
}

output_path = output_dir / "epo_intervals.pkl"
with open(output_path, "wb") as f:
    pickle.dump(intervals_dict, f, protocol=pickle.HIGHEST_PROTOCOL)

print(f"Saved intertimes to {output_path}")


Saved intertimes to ./outputs/epo_intervals.pkl
